# Routing optimization in a humanitarian context


We are considering real-world scenario of **distributing medical goods during disaster response**
following one of the worst tropical cyclones ever been recorded in Africa: **Cyclone Idai**.

In this scenario, a humanitarian organization shipped much needed medical goods to Beira, Mozambique, which were then
dispatched to local vehicles to be delivered across the region.
The supplies included vaccinations and medications for water-borne diseases such as Malaria and Cholera,
so distribution efficiency was critical to contain disastrous epidemics.


In [1]:
import folium
from folium.plugins import BeautifyIcon
import pandas as pd
import openrouteservice as ors

In [2]:
key = '5b3ce3597851110001cf6248084f60f6fc214724a79ddb5d419f214f'

## The logistics setup

In total 20 sites were identified in need of the medical supplies, while 3 vehicles were scheduled for delivery.
Let's assume there was only one type of goods, e.g. standard moving boxes full of one medication.
(In reality there were dozens of different good types, which can be modelled with the same workflow,
but that'd unnecessarily bloat this example).

The **vehicles** were all located in the port of Beira and had the same following constraints:

- operation time windows from 8:00 to 20:00
- loading capacity of 300 *[arbitrary unit]*

The **delivery locations** were mostly located in the Beira region, but some extended ~ 200 km to the north of Beira.
Their needs range from 10 to 148 units of the arbitrary medication goods


In [3]:
# First define the map centered around Beira
m = folium.Map(location=[-18.63680, 34.79430], tiles='cartodbpositron', zoom_start=8)

# Next load the delivery locations from CSV file at ../resources/data/idai_health_sites.csv
# ID, Lat, Lon, Open_From, Open_To, Needed_Amount
deliveries_data = pd.read_csv(
    'idai_health_sites.csv',
    index_col="ID",
    parse_dates=["Open_From", "Open_To"]
)

# Plot the locations on the map with more info in the ToolTip
for location in deliveries_data.itertuples():
    tooltip = folium.map.Tooltip("<h4><b>ID {}</b></p><p>Supplies needed: <b>{}</b></p>".format(
        location.Index, location.Needed_Amount
    ))

    folium.Marker(
        location=[location.Lat, location.Lon],
        tooltip=tooltip,
        icon=BeautifyIcon(
            icon_shape='marker',
            number=int(location.Index),
            spin=True,
            text_color='red',
            background_color="#FFF",
            inner_icon_style="font-size:12px;padding-top:-5px;"
        )
    ).add_to(m)

# The vehicles are all located at the port of Beira
depot = [-19.818474, 34.835447]

folium.Marker(
    location=depot,
    icon=folium.Icon(color="green", icon="bus", prefix='fa'),
    setZIndexOffset=1000
).add_to(m)

m

C:\Users\amitk\AppData\Local\Temp\ipykernel_29060\1617869977.py:6: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  deliveries_data = pd.read_csv(
C:\Users\amitk\AppData\Local\Temp\ipykernel_29060\1617869977.py:6: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  deliveries_data = pd.read_csv(


## The routing problem setup

Now that we have described the setup sufficiently, we can start to set up our actual Vehicle Routing Problem.
For this example we're using the FOSS library of [Vroom](https://github.com/VROOM-Project/vroom), which has
[recently seen](http://k1z.blog.uni-heidelberg.de/2019/01/24/solve-routing-optimization-with-vroom-ors/) support for
openrouteservice and is available through our APIs.

To properly describe the problem in algorithmic terms, we have to provide the following information:

- **vehicles start/end address**: vehicle depot in Beira's port
- **vehicle capacity**: 300
- **vehicle operational times**: 08:00 - 20:00
- **service location**: delivery location
- **service time windows**: individual delivery location's time window
- **service amount**: individual delivery location's needs



In [4]:
# Define the vehicles
# https://openrouteservice-py.readthedocs.io/en/latest/openrouteservice.html#openrouteservice.optimization.Vehicle
vehicles = list()
for idx in range(3):
    vehicles.append(
        ors.optimization.Vehicle(
            id=idx,
            start=list(reversed(depot)),
            # end=list(reversed(depot)),
            capacity=[300],
            time_window=[1553241600, 1553284800]  # Fri 8-20:00, expressed in POSIX timestamp
        )
    )

# Next define the delivery stations
# https://openrouteservice-py.readthedocs.io/en/latest/openrouteservice.html#openrouteservice.optimization.Job
deliveries = list()
for delivery in deliveries_data.itertuples():
    deliveries.append(
        ors.optimization.Job(
            id=delivery.Index,
            location=[delivery.Lon, delivery.Lat],
            service=1200,  # Assume 20 minutes at each site
            amount=[delivery.Needed_Amount],
            time_windows=[[
                int(delivery.Open_From.timestamp()),  # VROOM expects UNIX timestamp
                int(delivery.Open_To.timestamp())
            ]]
        )
    )

With that set up we can now perform the actual request and let openrouteservice calculate the optimal vehicle schedule
for all deliveries.

In [5]:
# Initialize a client and make the request
ors_client = ors.Client(key=key)  # Get an API key from https://openrouteservice.org/dev/#/signup
result = ors_client.optimization(
    jobs=deliveries,
    vehicles=vehicles,
    geometry=True
)

# Add the output to the map
for color, route in zip(['green', 'red', 'blue'], result['routes']):
    decoded = ors.convert.decode_polyline(route['geometry'])  # Route geometry is encoded
    gj = folium.GeoJson(
        name='Vehicle {}'.format(route['vehicle']),
        data={"type": "FeatureCollection", "features": [{"type": "Feature",
                                                         "geometry": decoded,
                                                         "properties": {"color": color}
                                                         }]},
        style_function=lambda x: {"color": x['properties']['color']}
    )
    gj.add_child(folium.Tooltip(
        """<h4>Vehicle {vehicle}</h4>
        <b>Distance</b> {distance} m <br>
        <b>Duration</b> {duration} secs
        """.format(**route)
    ))
    gj.add_to(m)

folium.LayerControl().add_to(m)
m

ApiError: 500 ({'code': 3, 'error': 'Invalid routing response: HTTP/1.1 503 Service Unavailable\r\nDate: Tue, 29 Apr 2025 18:16:01 GMT\r\nServer: Apache/2.4.54 (Unix)\r\nContent-Length: 299\r\nConnection: close\r\nContent-Type: text/html; charset=iso-8859-1\r\n\r\n<!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">\n<html><head>\n<title>503 Service Unavailable</title>\n</head><body>\n<h1>Service Unavailable</h1>\n<p>The server is temporarily unable to service your\nrequest due to maintenance downtime or capacity\nproblems. Please try again later.</p>\n</body></html>\n'})

## Data view

Plotting it on a map is nice, but let's add a little more context to it in form of data tables.
First the overall trip schedule:

### Overall schedule

In [ ]:
# Only extract relevant fields from the response
extract_fields = ['distance', 'amount', 'duration']
data = [{key: route[key] for key in extract_fields} for route in result['routes']]

vehicles_df = pd.DataFrame(data)
vehicles_df.index.name = 'vehicle'
vehicles_df

So every vehicle's capacity is almost fully exploited. That's good.
How about a look at the individual service stations:

In [ ]:
# Create a list to display the schedule for all vehicles
stations = list()
for route in result['routes']:
    vehicle = list()
    for step in route["steps"]:
        vehicle.append(
            [
                step.get("job", "Depot"),  # Station ID
                step["arrival"],  # Arrival time
                step["arrival"] + step.get("service", 0),  # Departure time

            ]
        )
    stations.append(vehicle)

Now we can look at each individual vehicle's timetable:

### Vehicle 0

In [ ]:
df_stations_0 = pd.DataFrame(stations[0], columns=["Station ID", "Arrival", "Departure"])
df_stations_0['Arrival'] = pd.to_datetime(df_stations_0['Arrival'], unit='s')
df_stations_0['Departure'] = pd.to_datetime(df_stations_0['Departure'], unit='s')
df_stations_0

### Vehicle 1

In [ ]:
df_stations_1 = pd.DataFrame(stations[1], columns=["Station ID", "Arrival", "Departure"])
df_stations_1['Arrival'] = pd.to_datetime(df_stations_1['Arrival'], unit='s')
df_stations_1['Departure'] = pd.to_datetime(df_stations_1['Departure'], unit='s')
df_stations_1

### Vehicle 2

In [ ]:
df_stations_2 = pd.DataFrame(stations[2], columns=["Station ID", "Arrival", "Departure"])
df_stations_2['Arrival'] = pd.to_datetime(df_stations_2['Arrival'], unit='s')
df_stations_2['Departure'] = pd.to_datetime(df_stations_2['Departure'], unit='s')
df_stations_2